# 环节 08 · Agentic RL 与信用分配（配套 Notebook）

> 配套长文：[环节08-AgenticRL与信用分配详解.md](./环节08-AgenticRL与信用分配详解.md)
> 定位：团队奖励的信用错配、组基线为何失效、价值分解 VDN、PRM 引导搜索、自博弈坍缩。全部**纯 Python 标准库**。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 信用错配 | §2 / §8.1 | 团队奖励会给"其实失败"的 agent 发正优势 |
| §2 组基线失效 | §2 / §8.2 | 7 条成功轨迹共享同一个优势 |
| §3 价值分解 | §3 / §8.3 | VDN 把团队价值拆成每人一份 |
| §4 PRM 与搜索 | §5 | 过程奖励能在"哪步开始歪"处给信号 |
| §5 自博弈坍缩 | §6 | 没有外部锚点，自我评估会漂移 |


## 1. 信用分配：团队奖励的错配

两个 agent 协作，奖励只看"团队是否成功"。于是**即使 a1 失败、只是 a2 兜住了**，a1 也会拿到正优势 → 错误强化。


In [ ]:
print(f"{'p1':>5} {'p2':>5} {'P(团队成功)':>12} {'成功里 a1 其实失败':>20}")
for p1, p2 in ((0.9, 0.9), (0.7, 0.7), (0.5, 0.5), (0.9, 0.3)):
    team = 1 - (1 - p1) * (1 - p2)                   # 团队奖励 = 至少一个成功
    mask = (1 - p1) * p2                             # a1 失败但 a2 兜住了
    print(f"{p1:>5.1f} {p2:>5.1f} {team:>12.3f} {mask / team * 100:>19.1f}%")
print("→ p=0.5/0.5 时，团队成功里有 33.3% 其实是 a1 失败却拿了正优势。")
print("  换成过程奖励（每步各自打分）→ 这个比例直接归 0。")


## 2. 组基线在多 agent 下失效

GRPO 的组基线给的是**团队级**优势：同一组里 7 条成功轨迹拿到**完全相同**的优势 —— 谁贡献了多少，完全没有区分。


In [ ]:
import statistics

rew = [1, 1, 0, 1, 1, 1, 1, 1]                       # 8 条团队轨迹的团队级奖励
mean, sd = statistics.fmean(rew), statistics.pstdev(rew)
advs = [(r - mean) / sd for r in rew]
print("团队级优势 =", [f"{a:+.2f}" for a in advs])
print("→ 7 条成功轨迹共享 +0.38：团队里「谁干了什么」没有被区分。")
print("  要区分就得有 per-agent 的价值估计 → 价值分解（下一节）。")


## 3. 价值分解（VDN）：`Q_total = Σ Q_i`

把团队 Q 值写成**每个 agent 各自 Q 的和**。好处：个体贪心 = 联合最优（可分解性），梯度能落到每个 agent 的动作上。


In [ ]:
Q = {(1, 0): 0.2, (1, 1): 0.9, (2, 0): 0.1, (2, 1): 0.7}   # (agent, action) -> Q_i

def q_total(a1, a2):
    return Q[(1, a1)] + Q[(2, a2)]

best = max(((a1, a2) for a1 in (0, 1) for a2 in (0, 1)), key=lambda ab: q_total(*ab))
greedy = [max((a for a in (0, 1)), key=lambda a: Q[(i, a)]) for i in (1, 2)]
print("团队最优联合动作 =", best, " Q_total =", round(q_total(*best), 3))
print("各 agent 的个体贪心 =", greedy)
print("→ 个体贪心拼起来就是联合最优：这就是 VDN 做信用分配的原理。")


## 4. PRM vs ORM：把信号摊到每一步

- **ORM**（结果奖励）：只看最后答对没 → 稀疏、晚、分不清"哪步开始歪"；
- **PRM**（过程奖励）：给每一步打分 → 能做**步骤级信用分配**，还能在**推理时**引导搜索（beam / tree search 用 PRM 选前缀）。


In [ ]:
paths = {                                          # 三条推理路径的过程分与最终分
    "A→B→(歪)→错": {"prm": [0.9, 0.2, 0.1], "orm": 0.0},
    "A→B→C→对":   {"prm": [0.9, 0.8, 0.9], "orm": 1.0},
    "A→(歪)→(更歪)": {"prm": [0.9, 0.3, 0.1], "orm": 0.0},
}
print(f"{'路径':>14} {'PRM 逐步':>16} {'PRM 均值':>9} {'ORM 终答':>9}")
for name, v in paths.items():
    print(f"{name:>14} {str(v['prm']):>16} {sum(v['prm']) / len(v['prm']):>9.3f} {v['orm']:>9.1f}")
print("→ 后两条 ORM 都是 0、分不开；PRM 却能看到「第二条一直稳、第三条第二步就歪了」。")
print("  把 PRM 当'打分器'做 beam search，就是推理时计算（test-time scaling）的一种。")


## 5. 自博弈的风险：没有外部锚点会漂移/坍缩

自博弈（SPIN / SPICE 类）用模型自己的判断当信号。若**没有外部真值锚定**，评价标准会随模型一起漂——自我强化到最后可能收敛到某个退化模式（模式崩塌）。


In [ ]:
import random

random.seed(0)
q, hist = 0.5, []
for _ in range(12):
    q += random.gauss(0, 0.08)                       # 无锚点：纯粹随机游走
    hist.append(round(q, 3))
print("无外部真值的自博弈（自我评分漂移）：")
print("  ", hist)
print("  min =", min(hist), " max =", max(hist), " 漂移幅度 =", round(max(hist) - min(hist), 3))
print("→ 缺锚点时会漂移或坍缩；SPIN/SPICE 用外部数据或可验证真值来锚定。")


## 6. 小结与下钻

- **多步/多 agent 的痛点是信用分配**：终答奖励太稀疏、太晚。
- **组基线在多 agent 下只给团队级信号**，要 per-agent 就得价值分解（VDN/QMIX）或 PRM。
- **PRM 既能训也能搜**：推理时用 PRM 引导 beam/tree search 是 test-time scaling 的一条线。
- **自博弈必须有锚点**，否则会漂移/模式崩塌。

返回：[环节00-总揽与环节导航](./环节00-总揽与环节导航.md)；工程总表见 [RL选型与工程落地总表](./RL选型与工程落地总表.md)。
